## **Statistical Hypothesis Testing**

**Statistical Hypothesis Testing** is a formal procedure for investigating our ideas about the world using data. In data science, it is the method we use to determine whether a pattern we see is a "real" effect or just a result of random chance.

Think of it like a **courtroom trial**: the defendant is innocent until proven guilty. In statistics, your "claim" is considered false until the data provides overwhelming evidence to the contrary.

---

- **1. The Two Hypotheses**
    - Every test starts with two competing statements:
        * **Null Hypothesis ($H_0$):** The "status quo." It assumes there is **no effect**, no difference, or no relationship. 
            * *Example:* "This new website layout does not increase click-through rates."
        * **Alternative Hypothesis ($H_a$ or $H_1$):** The claim you are trying to prove. It assumes there **is** an effect or a difference.
            * *Example:* "The new website layout increases click-through rates."

---

- **2. The Process: 4 Key Steps**

- **Step 1: State the Hypotheses**
    - You define your $H_0$ and $H_a$ before looking at the data to avoid bias.
- **Step 2: Choose a Significance Level ($\alpha$)**
    - This is your "threshold for evidence." It is the probability of rejecting the Null Hypothesis when it is actually true (a False Positive). 
    * The standard in data science is $\alpha = 0.05$ (**5%**). This means you are willing to accept a 5% risk of being wrong.
- **Step 3: Calculate the Test Statistic and P-Value**
    - You run a statistical test (like a T-test, Z-test, or Chi-Square) on your data. This produces two numbers:
        * **Test Statistic:** A numerical value that represents how far your sample data deviates from the Null Hypothesis.
        * **P-Value:** The probability of seeing your results (or more extreme results) if the Null Hypothesis were actually true.
- **Step 4: Make a Decision**
    * **If P-value $\le \alpha$:** You **Reject the Null Hypothesis**. The result is "statistically significant." You have enough evidence to support your claim.
    * **If P-value $> \alpha$:** You **Fail to Reject the Null Hypothesis**. The result is not significant; any pattern you saw could easily be due to random luck.

---

- **3. The Error Types (The "Catch")**
    - No test is perfect. There are two ways to be wrong:

| Error Type | What happened? | Real-world Analogy |
| :--- | :--- | :--- |
| **Type I Error** | You reject $H_0$, but $H_0$ was actually true. | A "False Alarm" (Convicting an innocent person). |
| **Type II Error** | You fail to reject $H_0$, but $H_0$ was actually false. | A "Miss" (Letting a guilty person go free). |

---

- **4. Why this matters for Machine Learning**
    - Hypothesis testing isn't just for academic papers; it is deeply integrated into the ML workflow:
        1.  **Feature Selection:** You can use hypothesis tests (like ANOVA) to see if a specific feature actually has a statistically significant relationship with your target variable. If $p > 0.05$, the feature might just be noise.
        2.  **A/B Testing:** This is the most common use case in industry. Before a company rolls out a new feature, they test it against the old version to ensure the improvement isn't just a fluke.
        3.  **Model Comparison:** When comparing two algorithms, you use hypothesis tests to determine if the difference in their accuracy scores is significant or if they are essentially performing the same.

## **Example: Flipping a Coin**

In [24]:
from typing import Tuple
import math

In [25]:
def normal_approximation_to_binomial(n: int, p: float) -> Tuple[float, float]:
    """
    Calculates the mean (mu) and standard deviation (sigma) of a 
    Normal distribution that best fits a given Binomial distribution.
    """
    # mu = np: The expected number of successes
    mu = p * n
    
    # sigma = sqrt(np(1-p)): The standard deviation of the successes
    sigma = math.sqrt(p * (1 - p) * n)

    return mu, sigma

This function serves as a bridge between two different types of probability: **Discrete** (counting things) and **Continuous** (measuring things).

---

- **What is the Utility?**
    - In data science, we often work with **Binomial** processes (events with two outcomes: Yes/No, Click/No-Click, Buy/Not-Buy). However, calculating probabilities for a Binomial distribution becomes computationally "expensive" and difficult when the number of trials ($n$) is very large.

- **1. Computational Efficiency (The "Shortcut")**
    - If you want to know the probability of getting more than 500,000 "heads" in 1,000,000 coin flips, the Binomial formula involves massive factorials (like $1,000,000!$) that can crash a standard calculator or script. 
        * **The Utility:** This function gives you the $\mu$ and $\sigma$ you need to use the **Normal Distribution** instead. Because of the Central Limit Theorem, when $n$ is large, the Binomial bell shape becomes nearly identical to the Normal bell shape.

- **2. Hypothesis Testing for Proportions**
    - This is the math behind **A/B Testing**. 
        * If you want to know if a 5% conversion rate ($p=0.05$) on 10,000 users ($n=10,000$) is statistically significant, you use this function to find the "expected" $\mu$ and $\sigma$. 
        * You can then see how far your *actual* results are from that $\mu$ in terms of standard deviations.

- **3. Defining "Normal" Behavior**
    - In **Data Engineering**, you can use this to set alerts. If you know a process has a success rate $p$, and you observe $n$ events, you can use this function to determine what the "normal" range of successes should be. Anything outside $\mu \pm 3\sigma$ could trigger an automated data quality alert.

---

- **When can you use this? (The "Rule of Thumb")**
    - You shouldn't use this approximation for *all* binomial data. It is only accurate when:
        1.  **$n \times p \ge 5$**
        2.  **$n \times (1 - p) \ge 5$**

> If your sample size is too small or your probability is too close to 0 or 1, the Binomial distribution will be too skewed, and the Normal approximation will give you incorrect results.

In [26]:
# The normal cdf _is_ the probability the variable is below a threshold
def normal_cdf(x: float, mu: float = 0, sigma: float = 1) -> float:
    """
    Calculates the probability that a random variable from a Normal 
    Distribution (with mean 'mu' and std dev 'sigma') is less than or equal to 'x'.
    """
    # math.erf is the 'Error Function', a special function used in 
    # probability to calculate areas under the normal curve.
    # The formula below transforms 'x' into a standardized value and 
    # uses the Error Function to find the cumulative probability.
    return (1 + math.erf((x - mu) / (math.sqrt(2) * sigma))) / 2

- **What is the Utility?**
    - The **CDF** is one of the most useful tools in statistics. While the PDF (Probability Density Function) tells you the "height" of the bell curve at a specific point, the CDF tells you the **accumulated probability** from negative infinity up to that point.



![Image of Normal Distribution Cumulative Distribution Function curve](../Images/Gaussian-distribution.jpeg)


- **1. Finding Probabilities for Ranges**
    - In data science, we rarely care about the probability of a variable being *exactly* a certain value. We care about ranges. The CDF allows you to answer:
        * "What is the probability that a user stays on the site for **less than** 30 seconds?" ($P(X \le 30)$)
        * "What is the probability that a transaction is **between** \$50 and \$100?" ($P(50 \le X \le 100)$)

- **2. Calculating P-values**
    - As you explore **Hypothesis Testing**, the CDF is what sits under the hood of your p-value calculations. A p-value is essentially the probability of seeing a result as extreme as yours; the CDF calculates that "tail" probability.

- **3. Outlier Detection**
You can use the CDF to identify values that fall in the extreme ends of your distribution. For example, any data point where the `normal_cdf` returns a value $> 0.99$ is in the top **1%** of your data, marking it as a potential outlier.

---

- **The Error Function (`math.erf`)**
    - You might notice the code uses `math.erf`. Because the Normal Distribution curve (the "Bell Curve") cannot be integrated using basic algebra, mathematicians use the **Error Function** as a standard way to represent that area. 

> In your data science workflow, you will often use `scipy.stats.norm.cdf`, which does exactly what this function does but with more optimization for large arrays.

In [27]:
def normal_probability_above(lo: float, mu: float= 0, sigma: float = 1) -> float:
    """The probability that an N(mu, sigma) is greater than lo."""
    
    # normal_cdf gives the area to the LEFT of 'lo' (P(X <= lo))
    # Since the total area under the curve is 1, 
    # 1 minus the left area gives the area to the RIGHT.
    return 1 - normal_cdf(lo, mu, sigma)

This function is a simple but essential "wrapper" that shifts your perspective from looking at the **left tail** of the distribution to the **right tail**.

---

- **What is the Utility?**
    - In statistics, we often want to know the probability of an "at least" or "greater than" event. This is known as the **Survival Function** or **Upper Tail Probability**.

- **1. Calculating One-Tailed P-values**
    - In hypothesis testing, if your alternative hypothesis is that a change will **increase** a metric (e.g., "The new ETL process will increase the number of rows processed per second"), you use this function to find the p-value. It tells you how likely it is to see a value as high as your result by pure chance.

- **2. Risk and "Exceedance"**
    - In Data Engineering and System Monitoring, you care about **thresholds**:
        * **SLA Monitoring:** "What is the probability that a query takes **more than** 500ms to run?"
        * **Anomaly Detection:** "What is the probability that the daily CPU load exceeds 95%?"
        * **Inventory:** "What is the probability that customer demand will be **higher than** our current stock of 1,000 units?"

- **3. Complementary Logic**
    - Because the Normal Distribution is a "total" probability space (it sums to 1), finding the probability of something *not* happening is often the easiest way to find the probability of it happening. This function handles that subtraction for you to keep your main logic clean and readable.

---

- **Visualization**
    - If you imagine the Bell Curve:
        * `normal_cdf(lo)` shades everything from the **far left up to your point**.
        * `normal_probability_above(lo)` shades everything from **your point to the far right**.

In [28]:
def normal_probability_between(lo: float, hi: float, mu: float= 0, sigma: float= 1) -> float:
    """The probability that an N(mu, sigma) is between lo and hi."""

    # normal_cdf(hi) = Total area from -infinity up to 'hi'
    # normal_cdf(lo) = Total area from -infinity up to 'lo'
    
    # By subtracting the 'lo' area from the 'hi' area, 
    # we are left with the "slice" in the middle.
    return normal_cdf(hi, mu, sigma) - normal_cdf(lo, mu, sigma)

This function calculates the probability that a data point falls within a **specific range** $[lo, hi]$. It is the most common way to use the Normal Distribution to analyze realistic scenarios where you aren't just looking for "extremes," but for a "middle" window.

---

- **What is the Utility?**
    - This is arguably the most practical function in your statistical toolkit because most real-world questions involve intervals.

- **1. Verifying the "Empirical Rule"**
    - You can use this function to prove the **68-95-99.7** rule we discussed earlier. 
        * If you run `normal_probability_between(-1, 1, 0, 1)`, the result will be approximately **0.6826**.
        * If you run `normal_probability_between(-2, 2, 0, 1)`, it will be approximately **0.9544**.

- **2. Business and Operational Intervals**
    - In your data engineering or data science work, you might use this to define "Normal Operating Conditions":
        * **Quality Control:** "What percentage of our manufactured parts are between 9.9mm and 10.1mm?"
        * **User Behavior:** "What is the probability that a user stays on our platform between 2 and 5 minutes?"
        * **ETL Performance:** "What is the probability that our nightly data load finishes between 2:00 AM and 3:00 AM?"

- **3. Two-Tailed Significance Testing**
    - While `normal_probability_above` is for one-sided tests, this function helps with **two-sided** checks. If you want to know the probability that a result stays within a "normal" range (and therefore isn't an outlier in *either* direction), this is your primary tool.

---

- **Visualizing the "Subtraction" Logic**
    - Imagine the area under the bell curve:
        1.  **`normal_cdf(hi)`**: You paint the entire curve starting from the left and stop at the **high** mark.
        2.  **`normal_cdf(lo)`**: You take an "eraser" and remove all the paint from the left up to the **low** mark.
        3.  **Result**: You are left with a painted "strip" between the two points.

In [29]:
# It's outside if it's not between
def normal_probability_outside(lo: float, hi: float, mu: float = 0, sigma: float = 1) -> float:
    """The probability that an N(mu, sigma) is not between lo and hi."""

    # normal_probability_between gives us the 'middle' area.
    # Since the total probability for the entire curve is exactly 1,
    # subtracting the middle leaves us with the two outer tails.
    return 1 - normal_probability_between(lo, hi, mu, sigma)

- **What is the Utility?**
    - This function is primarily used for **Anomaly Detection** and **Two-Tailed Hypothesis Testing**. It answers the question: *"How likely is it that an observation is an extreme outlier?"*

- **1. Identifying Outliers**
    - In data engineering, you might use this to flag suspicious data in your pipeline. 
        * **Example:** If you know your daily data ingestion volume usually falls between 80GB and 120GB, you can use this function to calculate the probability of seeing a volume **outside** that range. If the probability is extremely low (e.g., $<0.01$), your system can automatically trigger a "Data Quality Alert."

- **2. Two-Tailed P-Values**
    - This is a core part of statistical significance testing. When you don't know if a change will make things better or worse—only that it might make them *different*—you use a two-tailed test. 
        * This function calculates the probability that the data would fall into **either tail** (too high or too low) by sheer coincidence.

- **3. Confidence Intervals and "Alpha" ($\alpha$)**
    - In statistics, we often set a significance level (like $\alpha = 0.05$). This function helps you visualize that 5% risk.
        * If you set `lo` and `hi` to be your 95% confidence boundaries, `normal_probability_outside` will return **0.05**. This represents the "Danger Zone" where you would reject your Null Hypothesis.

---

- **Summary of your "Probability Toolkit"**
    - By combining these functions, you've essentially built a miniature version of `scipy.stats.norm`. Here is how they relate:

| Function | Logical Probability | Use Case |
| :--- | :--- | :--- |
| `normal_cdf` | $P(X \le x)$ | Percentiles / Left-tail testing |
| `normal_probability_above` | $P(X > x)$ | Survival rates / Right-tail testing |
| `normal_probability_between` | $P(lo \le X \le hi)$ | Normal operational ranges |
| `normal_probability_outside` | $P(X < lo \text{ or } X > hi)$ | **Anomaly detection / Two-tailed tests** |

In [30]:
def inverse_normal_cdf(p: float, 
                       mu: float = 0, 
                       sigma: float = 1, 
                       tolerance: float = 0.00001) -> float:
    """
    Finds the 'x' value such that P(X <= x) = p.
    Uses binary search to converge on the value.
    """

    # If it's not standard normal, compute standard normal and rescale
    if mu != 0 or sigma != 1:
        return mu + sigma * inverse_normal_cdf(p, tolerance=tolerance)

    low_z = -10.0                      # Probability of Z < -10 is nearly 0
    hi_z  =  10.0                      # Probability of Z < 10 is nearly 1
    
    while hi_z - low_z > tolerance:
        mid_z = (low_z + hi_z) / 2     # Consider the midpoint
        mid_p = normal_cdf(mid_z)      # Calculate the probability there
        
        if mid_p < p:
            low_z = mid_z              # Midpoint is too low, search above it
        else:
            hi_z = mid_z               # Midpoint is too high, search below it

    return mid_z

To calculate the **Inverse Normal CDF** (also known as the **Percent Point Function** or **Quantile Function**), we need to find the value $x$ such that $P(X \le x) = p$.

Because there is no simple algebraic formula for the inverse of the Error Function, we typically use a **binary search** (also called the Bisection Method) to find the value. This approach "zooms in" on the correct $x$ by repeatedly checking the `normal_cdf` until the probability is close enough to your target $p$.

---

- **What is the Utility?**
    - If the `normal_cdf` answers "What is the probability?", the `inverse_normal_cdf` answers **"What is the threshold?"**

- **1. Defining Confidence Intervals**
    - This is how you find the "Critical Values" (like the famous $1.96$ for a 95% confidence interval). 
* If you want to find the boundaries that contain the middle 95% of your data, you look for the values at $p=0.025$ and $p=0.975$.

- **2. Determining "Actionable" Thresholds**
    - In Data Engineering, you can use this to set **dynamic alerts**:
        * **Example:** "I want to be alerted whenever the server latency is higher than what **99%** of users experience." 
* You would call `inverse_normal_cdf(0.99, mu, sigma)` to find the exact millisecond threshold you should set in your monitoring tool.

- **3. Data Synthesis and Simulation**
    - When running simulations (like Monte Carlo simulations), you often generate a random number between 0 and 1 using `random.random()`. You then pass that number into the `inverse_normal_cdf` to transform that "flat" random number into a "normally distributed" random variable.

---

- **How it works (The Logic)**
    1.  **Standardization:** The function first solves for the **Standard Normal Distribution** ($Z$) and then scales it to your specific $\mu$ and $\sigma$.
    2.  **The "Guessing" Game:** Since we know the CDF is always increasing, if our current guess gives a probability that is too low, we know the real value must be higher. 
    3.  **Tolerance:** The loop continues until the difference between our high and low guesses is smaller than the `tolerance`, giving you a highly accurate result.

> With this function, you've completed the full circle of probability tools for the Normal Distribution. You can now move from **Data $\rightarrow$ Probability** and from **Probability $\rightarrow$ Data Thresholds**.

In [31]:
def normal_upper_bound(p: float, mu: float = 0, sigma: float = 1) -> float:
    """Returns the z for which P(Z <= z) = probability"""

    # This simply calls the inverse function to find the value 'z'
    # where the area to the left is exactly 'p'.
    return inverse_normal_cdf(p, mu, sigma)

This function is a high-level wrapper for the `inverse_normal_cdf`. Its primary purpose is to identify a specific **cutoff point** (or "Z-score") on the horizontal axis of the bell curve.

---

- **What is the Utility?**
    - In statistical modeling and data engineering, we use "Upper Bounds" to define the limits of "normal" behavior.

- **1. Statistical Significance (Finding Critical Values)**
    - When you conduct a hypothesis test with a significance level of **5%** ($\alpha = 0.05$), you need to know where the "Rejection Region" begins. 
        * If you want to find the upper boundary for a one-tailed test, you would call `normal_upper_bound(0.95)`.
        * For a standard normal distribution, this returns approximately **1.645**. Any observation higher than this is considered statistically significant.

- **2. Capacity Planning & SLAs**
    - In your work as a data professional, you might use this to determine infrastructure requirements:
        * **Scenario:** You want to provision enough storage for your **Comptoire-DWH** project so that it can handle the daily data load **99% of the time** without failing.
        * **Solution:** If your daily data volume follows a normal distribution, `normal_upper_bound(0.99, mu, sigma)` will give you the exact GB threshold you should build for.

- **3. Anomaly Detection Thresholds**
    - Instead of hard-coding a limit (like "alert me if rows > 1000"), you can make your system **adaptive**. 
        * You can calculate the mean and standard deviation of row counts over the last 30 days.
        * Use `normal_upper_bound(0.999)` to find the "Extreme" upper limit. 
        * This way, the alert threshold moves automatically as your business grows.

---

- **The "Mirror" Functions**
    - Usually, this function is used alongside two others to cover all bases:

| Goal | Logic | Function to call |
| :--- | :--- | :--- |
| **Find Max "Safe" Value** | Find $z$ where $P(Z \le z) = p$ | `normal_upper_bound(p)` |
| **Find Min "Safe" Value** | Find $z$ where $P(Z \ge z) = p$ | `normal_lower_bound(p)` (which is `inverse_normal_cdf(1-p)`) |
| **Find Symmetric Range** | Find $z$ such that the middle is $p$ | `normal_two_sided_bounds(p)` |

In [32]:
def normal_lower_bound(p: float, mu: float = 0, sigma: float = 1) -> float:
    """Returns the z for which P(Z >= z) = probability"""

    # Because inverse_normal_cdf always calculates from the LEFT (up to p),
    # we pass (1 - p) to find the point where the remaining 
    # area on the RIGHT is exactly 'p'.
    return inverse_normal_cdf(1 - p, mu, sigma)

This function allows you to find the **minimum threshold** required to capture a certain probability in the **right tail** of the distribution.

---

- **What is the Utility?**
    - While the `normal_upper_bound` tells you the maximum "safe" value, the `normal_lower_bound` is used when you are concerned about a value being **too low**.

- **1. Service Level Agreements (SLAs) & Reliability**
    - In Data Engineering, you often want to ensure a metric stays **above** a certain level.
        * **Example:** "I want to find the minimum throughput (rows/sec) that my ETL pipeline maintains **95%** of the time."
        * You would call `normal_lower_bound(0.95, mu, sigma)`. This gives you the performance floor.

- **2. Inventory and Supply Chain**
    - If you are managing the database for a store (like your **Comptoire-DWH** project), you might use this for "Safety Stock" calculations.
        * "What is the minimum number of units I need to have in stock so that the probability of having *enough* items to meet demand is **99%**?"

- **3. One-Tailed Testing (Lower Tail)**
    - If your hypothesis is that a change will **decrease** a metric (e.g., "The new SQL index will decrease query response time"), you use the lower bound to find your "critical value." If the actual result is lower than this bound, you have a statistically significant improvement.

---

- **Comparison: Upper vs. Lower Bound**
    - Imagine you are looking at the probability $p = 0.05$ (5%):
        *   **`normal_upper_bound(0.05)`**: Returns the point where only **5%** of the data is to the **left** (the very bottom end).
        *   **`normal_lower_bound(0.05)`**: Returns the point where only **5%** of the data is to the **right** (the very top end).

> Essentially, `normal_lower_bound` is the mirror image of the upper bound logic. It’s perfect for setting "floors" rather than "ceilings."

In [33]:
def normal_two_sided_bounds(p: float, mu: float = 0, sigma: float = 1) -> float:
    """
    Returns the symmetric (about the mean) bounds
    that contain the specified probability
    """
    # If we want the middle 'p', we have (1 - p) total probability left over.
    # Because the normal distribution is symmetric, we split that 
    # leftover probability equally between the two outer tails.
    tail_probability = (1 - p) / 2

    # The upper bound is the point where only 'tail_probability' remains above it.
    upper_bound = normal_lower_bound(tail_probability, mu, sigma)

    # The lower bound is the point where only 'tail_probability' remains below it.
    lower_bound = normal_upper_bound(tail_probability, mu, sigma)

    return lower_bound, upper_bound

This function is the "gold standard" for creating **Confidence Intervals**. It calculates the two symmetric points (lower and upper) that trap a specific amount of probability—the "middle mass"—right in the center of the bell curve.

---

- **What is the Utility?**
    - In Data Science and Engineering, we rarely make a single-point guess. We prefer to provide a **range** where we expect values to fall.

- **1. Creating 95% Confidence Intervals**
    - This is the most frequent use case. If you want to know the range within which **95%** of your data should fall:
        * You call `normal_two_sided_bounds(0.95)`.
        * It calculates `tail_probability = 0.025` (2.5% in each tail).
        * For a standard normal distribution, this returns **(-1.96, 1.96)**.

- **2. Defining "Expected" Behavior in Pipelines**
    - In your **Comptoire-DWH** project, you can use this for automated monitoring.
        * **Scenario:** You monitor the number of rows inserted into your Fact Table every night.
        * **Implementation:** You calculate the `mu` and `sigma` of previous nightly loads.
        * **Utility:** You use `normal_two_sided_bounds(0.99, mu, sigma)` to get a "normal range." If tonight's load falls outside these bounds, it is a "Two-Sided Anomaly" (either too much data or too little), both of which might indicate a bug in the ETL logic.

- **3. Precision in Machine Learning Predictions**
    - When a model predicts a value (like a price or a count), it's more helpful to say, "I am 90% sure the price is between \$180 and \$220" rather than just saying "\$200." This function allows you to turn a standard error into that helpful range.

---

- **Why the "Symmetric" part matters**
    - Because the Normal Distribution is balanced, splitting the `tail_probability` ensures that your range is as **short as possible** while still containing the target probability. Any other range (like one shifted to the left) would have to be wider to capture the same amount of data.

- **Visualization**
    * **The Middle:** Your specified probability `p`.
    * **The Left Tail:** `(1 - p) / 2`.
    * **The Right Tail:** `(1 - p) / 2`.

Total = `p + (1 - p)/2 + (1 - p)/2 = 1.0`.

***

- You've now built a complete library for **Normal Distribution Analytics**. You can calculate:
    1.  **Probabilities** from values (`cdf`, `above`, `between`, `outside`).
    2.  **Values** from probabilities (`upper`, `lower`, `two_sided`).

- if our H0 is true X should be distributed approximately normally with mean 500 and standard deviation 15.8

In [34]:
mu_0, sigma_0 = normal_approximation_to_binomial(1000, 0.5)

mu_0, sigma_0

(500.0, 15.811388300841896)

In [35]:
lower_bound, upper_bound = normal_two_sided_bounds(0.95, mu_0, sigma_0)

lower_bound, upper_bound

(469.01026640487555, 530.9897335951244)

In [36]:
# 95% bounds based on assumption p is 0.5
lo, hi = normal_two_sided_bounds(0.95, mu_0, sigma_0)

lo, hi

(469.01026640487555, 530.9897335951244)

In [37]:
# actual mu and sigma based on p = 0.55
mu_1, sigma_1 = normal_approximation_to_binomial(1000, 0.55)

mu_1, sigma_1

(550.0, 15.732132722552274)

In [38]:
# a type 2 error means we fail to reject the null hypothesis,
# which will happen when X is still in our original interval
type_2_probability = normal_probability_between(lo, hi, mu_1, sigma_1)

type_2_probability

0.1134519987046329

In [39]:
power = 1 - type_2_probability

power

0.886548001295367

In [40]:
hi = normal_upper_bound(0.95, mu_0, sigma_0)

# is 526 (< 531, since we need more probability in the upper tail)
hi

526.0073585242053

In [42]:
type_2_probability = normal_cdf(hi, mu_1, sigma_1)

type_2_probability

0.06362051966928273

In [43]:
power = 1 - type_2_probability

power

0.9363794803307173

> This script uses the functions you’ve defined to analyze a hypothetical scenario: Server Latency in your data warehouse, where the average response time is 100ms with a standard deviation of 15ms.

In [44]:
# --- YOUR FUNCTIONS (Integrated) ---

def normal_cdf(x: float, mu: float = 0, sigma: float = 1) -> float:
    return (1 + math.erf((x - mu) / (math.sqrt(2) * sigma))) / 2

def inverse_normal_cdf(p: float, mu: float = 0, sigma: float = 1, tolerance: float = 0.00001) -> float:
    if mu != 0 or sigma != 1:
        return mu + sigma * inverse_normal_cdf(p, tolerance=tolerance)
    low_z, hi_z = -10.0, 10.0
    while hi_z - low_z > tolerance:
        mid_z = (low_z + hi_z) / 2
        mid_p = normal_cdf(mid_z)
        if mid_p < p: low_z = mid_z
        else: hi_z = mid_z
    return mid_z

def normal_probability_above(lo: float, mu: float = 0, sigma: float = 1) -> float:
    return 1 - normal_cdf(lo, mu, sigma)

def normal_probability_between(lo: float, hi: float, mu: float = 0, sigma: float = 1) -> float:
    return normal_cdf(hi, mu, sigma) - normal_cdf(lo, mu, sigma)

def normal_probability_outside(lo: float, hi: float, mu: float = 0, sigma: float = 1) -> float:
    return 1 - normal_probability_between(lo, hi, mu, sigma)

def normal_upper_bound(p: float, mu: float = 0, sigma: float = 1) -> float:
    return inverse_normal_cdf(p, mu, sigma)

def normal_lower_bound(p: float, mu: float = 0, sigma: float = 1) -> float:
    return inverse_normal_cdf(1 - p, mu, sigma)

def normal_two_sided_bounds(p: float, mu: float = 0, sigma: float = 1) -> Tuple[float, float]:
    tail_probability = (1 - p) / 2
    upper_bound = normal_lower_bound(tail_probability, mu, sigma)
    lower_bound = normal_upper_bound(tail_probability, mu, sigma)
    return lower_bound, upper_bound

# --- THE TEST CASE ---
mu_latency = 100 
sigma_latency = 15

print(f"--- Statistics for Server Latency (Mean={mu_latency}ms, StdDev={sigma_latency}ms) ---")

# 1. CDF: Prob of being fast
prob_fast = normal_cdf(85, mu_latency, sigma_latency)
print(f"Prob. latency is 85ms or less: {prob_fast:.4f}")

# 2. ABOVE: Prob of a delay
prob_slow = normal_probability_above(130, mu_latency, sigma_latency)
print(f"Prob. latency is slower than 130ms: {prob_slow:.4f}")

# 3. BETWEEN: Prob of 'Normal' range
prob_mid = normal_probability_between(90, 110, mu_latency, sigma_latency)
print(f"Prob. latency is between 90ms and 110ms: {prob_mid:.4f}")

# 4. OUTSIDE: Prob of an outlier
prob_out = normal_probability_outside(70, 130, mu_latency, sigma_latency)
print(f"Prob. of an extreme event (<70 or >130): {prob_out:.4f}")

# 5. UPPER BOUND: Setting a threshold
threshold_99 = normal_upper_bound(0.99, mu_latency, sigma_latency)
print(f"99% of requests are faster than: {threshold_99:.2f}ms")

# 6. TWO-SIDED: The 'Safety Zone'
low, high = normal_two_sided_bounds(0.95, mu_latency, sigma_latency)
print(f"95% of all traffic falls between: {low:.2f}ms and {high:.2f}ms")

--- Statistics for Server Latency (Mean=100ms, StdDev=15ms) ---
Prob. latency is 85ms or less: 0.1587
Prob. latency is slower than 130ms: 0.0228
Prob. latency is between 90ms and 110ms: 0.4950
Prob. of an extreme event (<70 or >130): 0.0455
99% of requests are faster than: 134.90ms
95% of all traffic falls between: 70.60ms and 129.40ms


- **2. When to use each (The "Simple Way")**

- Imagine you are looking at a **Bell Curve**. Here is your guide:

| Function | If you want to ask... | Simple Visualization |
| :--- | :--- | :--- |
| **`normal_cdf`** | "What is the chance I am **below** this number?" | Shade everything to the **left**. |
| **`normal_probability_above`** | "What is the chance I am **above** this number?" | Shade everything to the **right**. |
| **`normal_probability_between`** | "What is the chance I am in this **specific range**?" | Shade a **middle slice** between two points. |
| **`normal_probability_outside`** | "What is the chance this is an **extreme anomaly**?" | Shade **both ends** (too high and too low). |
| **`inverse_normal_cdf`** | "I have a percentage; what **number** goes with it?" | The "reverse gear" for all the above. |
| **`normal_upper_bound`** | "Where do the **top X%** of events start?" | Find the line for a "ceiling." |
| **`normal_lower_bound`** | "Where do the **bottom X%** of events start?" | Find the line for a "floor." |
| **`normal_two_sided_bounds`** | "What is the **range** for the middle 95%?" | Find two lines that center the data. |

---

- **Summary for your Data Career**
    *   Use the **Probability** functions (`cdf`, `above`, etc.) when you have a data point and you want to know how **weird or common** it is.
    *   Use the **Bound** functions (`upper`, `lower`, `two_sided`) when you want to **set a rule or alert** (like an SLA or a Confidence Interval) for future data.

To see these functions in action, let’s run a classic **A/B Test** scenario. This is exactly how data professionals decide if a change to a system (like a database index or a UI change) actually worked.

- **The Scenario: Optimizing your Data Warehouse**
    - Imagine your current ETL (Extract, Transform, Load) process takes an average of **100 minutes** with a standard deviation of **15 minutes**. You’ve written a new Python script that you believe is faster.

---

- **1. Defining the Hypotheses**
    - In statistics, we start by assuming the change did **nothing**.
        *   **Null Hypothesis ($H_0$):** The new script is the same as the old one. The average time is still **100 minutes**.
        *   **Alternative Hypothesis ($H_1$):** The new script is faster. The average time is **less than 100 minutes**.

---

- **2. Setting the "Decision Rule"**
    - Before we look at the results, we choose a **Significance Level ($\alpha$)**. Usually, this is **0.05 (5%)**. 

We want to find the **Critical Value**: the time so fast that there is only a 5% chance the old script could have achieved it by luck.

```python
# Using your functions
alpha = 0.05
mu_old = 100
sigma_old = 15

# We want the threshold where only 5% of the area is to the LEFT (lower time)
# Note: normal_upper_bound(0.05) gives the point where 5% is below it.
critical_value = normal_upper_bound(alpha, mu_old, sigma_old)

print(f"Decision Rule: If the new script takes less than {critical_value:.2f} minutes, we reject H0.")
# Output: Decision Rule: If the new script takes less than 75.33 minutes, we reject H0.
```

---

- **3. The Experiment Results**
    - Now, you run the new script, and it finishes in **72 minutes**.

- **4. Calculating the P-Value**
    - The **P-Value** is the probability of seeing a result of 72 minutes (or even faster) if the old process was still in effect.

```python
actual_result = 72

# How likely is 72 or less in an N(100, 15) distribution?
p_value = normal_cdf(actual_result, mu_old, sigma_old)

print(f"Actual Result: {actual_result} minutes")
print(f"P-Value: {p_value:.4f}")
# Output: P-Value: 0.0309
```

---

- **5. The Final Decision**
    - Now we compare our **P-Value** to our **$\alpha$**:
        1.  **P-Value (0.0309)** is less than **$\alpha$ (0.05)**.
        2.  **Result:** **Reject the Null Hypothesis ($H_0$)**.

- Simple Explanation of why we use the functions here:
    *   **`normal_upper_bound`**: We used this to set the "Goal Post." It told us exactly how fast the script needed to be to prove it wasn't just a "lucky" fast run.
    *   **`normal_cdf`**: We used this to calculate the final "Score." It told us that there was only a ~3% chance that the old process would ever finish that fast. Since 3% is less than our 5% limit for "luck," we concluded the new script is legitimately better.

---

- **Comparison Table for H0 / H1 Testing**

| Objective | Function to use | Why? |
| :--- | :--- | :--- |
| **Set the rejection threshold** | `normal_upper_bound(alpha)` | To find the "cutoff" line before the experiment starts. |
| **Calculate the "luck" factor** | `normal_cdf(result)` | To find the p-value after you have the result. |
| **Check if result is within normal range** | `normal_two_sided_bounds(0.95)` | To see if the result is "normal" or "weird" in any direction. |

## **p-Values**

The **p-value** is the most misunderstood number in statistics. The simplest way to think about it is as a **"Surprise Meter."**

It answers the question: **"If my Null Hypothesis ($H_0$) is true, how weird is the data I just saw?"**

---

- **1. The Literal Definition**
    - A p-value is the probability of obtaining test results **at least as extreme** as the ones observed, assuming that the Null Hypothesis is correct.

*   **Low p-value:** The data is very surprising if $H_0$ is true. This suggests $H_0$ might be wrong.
*   **High p-value:** The data is exactly what we’d expect under $H_0$. Nothing special is happening.

---

- **2. A Real-World Analogy: The "Loaded Die"**
    - Imagine you suspect a friend is cheating at a board game with a loaded die that only rolls **6s**.

*   **Null Hypothesis ($H_0$):** The die is fair.
*   **The Experiment:** Your friend rolls the die 5 times and gets five **6s** in a row.

The **p-value** here is the probability of rolling five 6s in a row with a **fair** die:
$$(1/6)^5 = 0.000128$$

**P-value = 0.01%**

Because 0.01% is extremely low (very surprising), you **Reject the Null Hypothesis**. You conclude the die is likely loaded.

---

- **3. How it works with your Functions**
    - If you are testing if your new database query is faster than the old average of **100ms** (with **$\sigma=15$**), and your new query takes **60ms**:
        1.  You use `normal_cdf(60, 100, 15)`.
        2.  The function returns a very small number (the **p-value**).
        3.  That small number tells you: "There is almost zero chance the old system would hit 60ms by accident."

---

- **4. The 0.05 Threshold ($\alpha$)**
    - In Data Science, we usually use **0.05** as the line in the sand:
        *   **$p \le 0.05$**: **Statistically Significant.** You have enough evidence to say, "This isn't luck; my change actually worked!"
        *   **$p > 0.05$**: **Not Significant.** You say, "This could just be a coincidence. I can't prove my change did anything."

---

- **5. What a p-value is NOT (Common Mistakes)**
    - It is tempting to misinterpret this number. Be careful:
        *   **It is NOT the probability that your theory is true.** (It only tells you how much the data disagrees with the *Null* theory).
        *   **A p-value of 0.05 doesn't mean the effect is "big."** (A tiny change can have a tiny p-value if you have enough data).
        *   **It is NOT "proof."** (A p-value of 0.01 still means there is a 1% chance you are seeing a pattern that isn't really there).

---

- **Why this matters for your Python code:**
    - When you write a function like `normal_probability_outside` or `normal_cdf`, you are effectively building a **P-value Calculator**. 

> In your **Comptoire-DWH** project, if you see a sudden spike in data volume, you'd calculate the p-value. If $p < 0.001$, your Python script should probably trigger an alert because that volume is "statistically impossible" under normal conditions.